In [ ]:
import requests
import os
import random
import datetime
from urllib.parse import unquote, quote


def generate_random_duration():
    duration_seconds = random.randint(30, 300)
    minutes = duration_seconds // 60
    seconds = duration_seconds % 60
    return f"{minutes:02}:{seconds:02}"


def escape_sql_string(value):
    return value.replace("'", "''")


def escape_ampersand(url):
    if "&" in url:
        return url.replace("&", "%26")
    else:
        return url


def fetch_bitbucket_files(repo_url):
    # Ekstrak informasi pemilik, repository, branch, dan path folder dari URL
    parts = repo_url.split("/")
    owner = parts[3]
    repo = parts[4]
    branch = parts[6]
    folder_path = "/".join(parts[7:])

    base_time = datetime.datetime.now().replace(microsecond=0)

    insert_statement_template = "INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES \n"

    # Parameters for the static fields
    # 1 = IndoPride
    # 2 = 日本の歌
    # 3 = Javasong
    # 4 = Worldwide
    category = "2"
    artist = "Go Shiina / Yuki Kajiura"
    album = "Demon Slayer Kimetsu no Yaiba Hashira Training Arc Vol.2 Bonus CD Part 2"
    cover_link = "https://bitbucket.org/sibeux/install-application/raw/319942f887cb294373a5091e90aee4cfeaf58be2/KNY-S4%20Vol.2%20Bonus%20CD/cover.jpg"
    favorite = "0"

    values = []
    index = 0

    while repo_url:
        # Buat URL API untuk folder
        api_url = f"https://api.bitbucket.org/2.0/repositories/{owner}/{repo}/src/{branch}/{folder_path}?pagelen=100"
        # print(f"Fetching URL: {api_url}")

        response = requests.get(api_url)

        if response.status_code == 200:
            files = response.json().get('values', [])
            for file in files:
                if file["type"] == "commit_file":
                    file_name_without_ext = os.path.splitext(
                        file["path"].split("/")[-1])[0]

                    date_added = base_time + datetime.timedelta(seconds=index)
                    date_added_str = date_added.strftime("%Y-%m-%d %H:%M:%S")

                    index += 1

                    duration = generate_random_duration()

                    # URL encode untuk menghindari masalah dengan karakter khusus
                    encoded_url = quote(
                        file['links']['self']['href'], safe=':/')

                    # Unquote untuk menghapus encoding berlebih
                    clean_url = unquote(encoded_url)
                    # Escape karakter '&' di URL jika ada
                    clean_url = escape_ampersand(clean_url)

                    artist = escape_sql_string(artist)
                    album = escape_sql_string(album)
                    cover_link = escape_sql_string(cover_link)
                    date_added_str = escape_sql_string(date_added_str)

                    values.append(
                        f"(NULL, '{category}', '{clean_url}', '{file_name_without_ext}', '{artist}', '{album}', '{duration}', '{cover_link}', '{favorite}', '{date_added_str}')"
                    )

            # Cek apakah ada halaman berikutnya
            repo_url = response.json().get('next')
        else:
            print("Error:", response.status_code, response.json())
            break

    insert_statement = insert_statement_template + ",\n".join(values) + ";"
    print(insert_statement)


# Contoh penggunaan dengan URL Bitbucket
repo_url = "https://bitbucket.org/sibeux/kirito-asuna/src/main/crossing%20field/"
fetch_bitbucket_files(repo_url)

INSERT INTO music (id_music, category, link_gdrive, title, artist, album, time, cover, favorite, date_added) VALUES 
(NULL, '2', 'https://api.bitbucket.org/2.0/repositories/sibeux/neon-genesis/src/b815fea36ff0c9e73831b61ed87455515ee2ef7c/Inuyasha%20Music%20Coll%201/.here.txt', '.here', 'Go Shiina / Yuki Kajiura', 'Demon Slayer Kimetsu no Yaiba Hashira Training Arc Vol.2 Bonus CD Part 2', '01:54', 'https://bitbucket.org/sibeux/install-application/raw/319942f887cb294373a5091e90aee4cfeaf58be2/KNY-S4%20Vol.2%20Bonus%20CD/cover.jpg', '0', '2025-05-26 12:12:41'),
(NULL, '2', 'https://api.bitbucket.org/2.0/repositories/sibeux/neon-genesis/src/b815fea36ff0c9e73831b61ed87455515ee2ef7c/Inuyasha%20Music%20Coll%201/80371-1535880890.jpg', '80371-1535880890', 'Go Shiina / Yuki Kajiura', 'Demon Slayer Kimetsu no Yaiba Hashira Training Arc Vol.2 Bonus CD Part 2', '01:58', 'https://bitbucket.org/sibeux/install-application/raw/319942f887cb294373a5091e90aee4cfeaf58be2/KNY-S4%20Vol.2%20Bonus%20CD/cover.jp